# Step 8 - Feature Group Ablation And Importance

This notebook checks which feature groups are doing the most work in the current model.


In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().resolve().parent))

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## Load Flights And Weather


In [2]:
from src.config import AIRPORT_TIMEZONES
from src.data import load_flights, load_weather
from src.features import (
    prepare_flights,
    add_scheduled_departure_utc,
    prepare_weather_features,
    add_delay_rate_feature,
    add_rolling_delay_features,
)
from src.weather_join import join_weather_to_flights

flights = prepare_flights(load_flights())
weather = load_weather()

flights = flights[flights["Origin"].isin(sorted(weather["station"].dropna().unique()))].copy()
flights = add_scheduled_departure_utc(flights, AIRPORT_TIMEZONES)
flights = flights.dropna(subset=["scheduled_departure_utc"]).copy()

model_df = join_weather_to_flights(flights, weather, tolerance_hours=2)
model_df = prepare_weather_features(model_df)
model_df = model_df.sort_values("scheduled_departure_utc").copy()

model_df = add_rolling_delay_features(model_df, ["Reporting_Airline"], "airline")
model_df = add_rolling_delay_features(model_df, ["Reporting_Airline", "Origin"], "airline_origin")

rolling_feature_cols = [
    "airline_delay_rate_prev_3h",
    "airline_delay_count_prev_3h",
    "airline_flight_count_prev_3h",
    "airline_origin_delay_rate_prev_3h",
    "airline_origin_delay_count_prev_3h",
    "airline_origin_flight_count_prev_3h",
]
for col in rolling_feature_cols:
    model_df[col] = model_df[col].fillna(0)

train_df = model_df[model_df["FlightDate"].dt.year == 2022].copy()
test_df = model_df[model_df["FlightDate"].dt.year == 2023].copy()

train_df, test_df = add_delay_rate_feature(train_df, test_df, "Origin", "Delay", "origin_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "Reporting_Airline", "Delay", "airline_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "route", "Delay", "route_delay_rate")

print(train_df.shape, test_df.shape)


(3398442, 45) (3519506, 45)


## Feature Groups


In [3]:
time_numeric = ["DayOfWeek", "dep_hour", "is_weekend"]
static_delay_numeric = ["origin_delay_rate", "airline_delay_rate", "route_delay_rate"]
weather_numeric = ["tmpf", "relh", "sknt", "alti", "vsby", "weather_report_age_minutes", "p01i"]
rolling_airline_numeric = rolling_feature_cols

route_categorical = ["Reporting_Airline", "Origin", "Dest", "route", "time_of_day_bin"]

feature_sets = {
    "All Features": {
        "numeric": time_numeric + static_delay_numeric + weather_numeric + rolling_airline_numeric,
        "categorical": route_categorical,
    },
    "No Weather": {
        "numeric": time_numeric + static_delay_numeric + rolling_airline_numeric,
        "categorical": route_categorical,
    },
    "No Rolling Airline": {
        "numeric": time_numeric + static_delay_numeric + weather_numeric,
        "categorical": route_categorical,
    },
    "No Static Delay Rates": {
        "numeric": time_numeric + weather_numeric + rolling_airline_numeric,
        "categorical": route_categorical,
    },
    "Time And Route Only": {
        "numeric": time_numeric,
        "categorical": route_categorical,
    },
}


## Group Ablation


In [4]:
def build_pipeline(numeric_features, categorical_features):
    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="most_frequent"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=150,
            max_depth=14,
            min_samples_leaf=5,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])


results = []
saved = {}

for name, spec in feature_sets.items():
    features = spec["numeric"] + spec["categorical"]
    X_train = train_df[features]
    y_train = train_df["Delay"]
    X_test = test_df[features]
    y_test = test_df["Delay"]

    model = build_pipeline(spec["numeric"], spec["categorical"])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "feature_set": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_delay": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_delay": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_delay": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    })
    saved[name] = model

ablation_results = pd.DataFrame(results).sort_values("f1_delay", ascending=False)
ablation_results


,feature_set,accuracy,precision_delay,recall_delay,f1_delay
3,No Static Delay Rates,0.700250,0.372950,0.620555,0.465898
0,All Features,0.699148,0.371983,0.621874,0.465513
1,No Weather,0.697853,0.370281,0.619672,0.463563
2,No Rolling Airline,0.641952,0.314685,0.593930,0.411398
4,Time And Route Only,0.592346,0.290655,0.649066,0.401512


## Feature Importance For The Best Set


In [5]:
best_name = ablation_results.iloc[0]["feature_set"]
best_model = saved[best_name]

preprocessor = best_model.named_steps["preprocessor"]
feature_names = preprocessor.get_feature_names_out()
importances = best_model.named_steps["classifier"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_df.head(25)


,feature,importance
0,num__airline_origin_delay_rate_prev_3h,0.205386
1,num__airline_delay_rate_prev_3h,0.173002
2,num__airline_delay_count_prev_3h,0.112113
3,num__airline_origin_delay_count_prev_3h,0.106322
4,num__dep_hour,0.085483
5,cat__time_of_day_bin_morning,0.058532
6,cat__Reporting_Airline_WN,0.022866
7,cat__time_of_day_bin_evening,0.021193
8,num__p01i,0.014476
9,num__airline_origin_flight_count_prev_3h,0.013449
